<a href="https://colab.research.google.com/github/clee2026/MSDS_498_Capstone/blob/main/nlp_categories/01_nyc311_nlp_issue_category_discovery.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 0. Install packages
!pip install -q pyarrow pandas scikit-learn sentence-transformers tqdm

In [ ]:
# 1. Mount Google Drive

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 2. Configuration
# Change these paths and settings as needed.

import os
from pathlib import Path

PARQUET_PATH = "/content/drive/MyDrive/project_data/Capstone Course Project Files/Data/nyc_311_FINAL_UPDATED.parquet"

OUTPUT_DIR = "/content/nlp_issue_outputs"
# OUTPUT_DIR = "/content/drive/MyDrive/project_data/Capstone Course Project Files/outputs_nlp_issue_categories"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

MAX_ROWS_TO_SAMPLE = 300_000
PARQUET_BATCH_SIZE = 50_000

# Candidate text columns.
# The notebook will only use the columns that actually exist in the parquet.
CANDIDATE_TEXT_COLUMNS = [
    "complaint_type",
    "descriptor",
    "resolution_description",
    "incident_zip",
    "borough",
    "agency_name",
    "location_type"
]

# Best starting value. You can rerun with 10, 15, 20, 25, etc.
NUM_CLUSTERS = 20

RANDOM_STATE = 42

In [ ]:
from pathlib import Path
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

In [ ]:
# 3. Inspect parquet schema

import pyarrow.parquet as pq
import pandas as pd

pf = pq.ParquetFile(PARQUET_PATH)
parquet_columns = pf.schema.names

print("Number of parquet columns:", len(parquet_columns))
print("First 50 columns:")
print(parquet_columns[:50])

available_text_columns = [c for c in CANDIDATE_TEXT_COLUMNS if c in parquet_columns]

print("\nAvailable candidate text columns:")
print(available_text_columns)

if not available_text_columns:
    raise ValueError("None of the candidate text columns were found. Update CANDIDATE_TEXT_COLUMNS after reviewing parquet_columns.")

Number of parquet columns: 46
First 50 columns:
['address_type', 'agency', 'agency_name', 'bbl', 'borough', 'bridge_highway_direction', 'bridge_highway_name', 'bridge_highway_segment', 'city', 'closed_date', 'community_board', 'complaint_type', 'council_district', 'created_date', 'cross_street_1', 'cross_street_2', 'descriptor', 'descriptor_2', 'due_date', 'facility_type', 'incident_address', 'incident_zip', 'intersection_street_1', 'intersection_street_2', 'landmark', 'latitude', 'location', 'location_type', 'longitude', 'open_data_channel_type', 'park_borough', 'park_facility_name', 'police_precinct', 'resolution_action_updated_date', 'resolution_description', 'road_ramp', 'source_file', 'source_parquet', 'status', 'street_name', 'taxi_company_borough', 'taxi_pick_up_location', 'unique_key', 'vehicle_type', 'x_coordinate_state_plane', 'y_coordinate_state_plane']

Available candidate text columns:
['complaint_type', 'descriptor', 'resolution_description', 'incident_zip', 'borough', 'a

In [ ]:
# 4. Load a sample from the 20M record parquet

from tqdm.auto import tqdm

dfs = []
rows_loaded = 0

for batch in tqdm(pf.iter_batches(batch_size=PARQUET_BATCH_SIZE, columns=available_text_columns)):
    df_batch = batch.to_pandas()
    dfs.append(df_batch)
    rows_loaded += len(df_batch)

    if rows_loaded >= MAX_ROWS_TO_SAMPLE:
        break

df = pd.concat(dfs, ignore_index=True).head(MAX_ROWS_TO_SAMPLE)

print("Loaded sample shape:", df.shape)
display(df.head())

0it [00:00, ?it/s]

Loaded sample shape: (300000, 7)


,complaint_type,descriptor,resolution_description,incident_zip,borough,agency_name,location_type
0,Street Condition,Pothole,The Department of Transportation referred this...,11378.0,QUEENS,Department of Transportation,<NA>
1,Street Condition,Pothole,The Department of Transportation determined th...,11377.0,QUEENS,Department of Transportation,<NA>
2,Street Condition,Pothole,The Department of Transportation referred this...,11416.0,QUEENS,Department of Transportation,<NA>
3,Graffiti,Graffiti,The graffiti on this property has been schedul...,10003.0,MANHATTAN,Department of Sanitation,Mixed Use
4,Street Condition,Pothole,The Department of Transportation determined th...,11237.0,BROOKLYN,Department of Transportation,<NA>


In [ ]:
# 5. Build combined text field

# Heavier weight is given to complaint_type and descriptor because those fields usually
# describe the issue more directly than resolution_description.

def safe_col(name):
    if name in df.columns:
        return df[name].fillna("").astype(str)
    return ""

combined_parts = []

if "complaint_type" in df.columns:
    combined_parts.append(safe_col("complaint_type"))
if "descriptor" in df.columns:
    combined_parts.append(safe_col("descriptor"))
if "resolution_description" in df.columns:
    combined_parts.append(safe_col("resolution_description"))

# Optional context fields. These should not dominate the category, but can help interpret issues.
for optional_col in ["agency_name", "location_type", "borough"]:
    if optional_col in df.columns:
        combined_parts.append(safe_col(optional_col))

df["combined_text"] = combined_parts[0]
for part in combined_parts[1:]:
    df["combined_text"] = df["combined_text"] + " " + part

df["combined_text"] = df["combined_text"].str.strip()
df = df[df["combined_text"] != ""].copy()

print("Rows with usable text:", df.shape[0])
display(df[available_text_columns + ["combined_text"]].head())

Rows with usable text: 300000


,complaint_type,descriptor,resolution_description,incident_zip,borough,agency_name,location_type,combined_text
0,Street Condition,Pothole,The Department of Transportation referred this...,11378.0,QUEENS,Department of Transportation,<NA>,Street Condition Pothole The Department of Tra...
1,Street Condition,Pothole,The Department of Transportation determined th...,11377.0,QUEENS,Department of Transportation,<NA>,Street Condition Pothole The Department of Tra...
2,Street Condition,Pothole,The Department of Transportation referred this...,11416.0,QUEENS,Department of Transportation,<NA>,Street Condition Pothole The Department of Tra...
3,Graffiti,Graffiti,The graffiti on this property has been schedul...,10003.0,MANHATTAN,Department of Sanitation,Mixed Use,Graffiti Graffiti The graffiti on this propert...
4,Street Condition,Pothole,The Department of Transportation determined th...,11237.0,BROOKLYN,Department of Transportation,<NA>,Street Condition Pothole The Department of Tra...


In [ ]:
# 6. Clean text

import re

DOMAIN_STOPWORDS = {
    "please", "call", "called", "complaint", "customer", "service",
    "request", "reported", "information", "provided", "will", "may",
    "nyc", "city", "department"
}

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    tokens = [t for t in text.split() if len(t) > 2 and t not in DOMAIN_STOPWORDS]
    return " ".join(tokens)

df["clean_text"] = df["combined_text"].apply(clean_text)
df = df[df["clean_text"].str.len() > 5].copy()

print("Rows after cleaning:", df.shape[0])
display(df[["combined_text", "clean_text"]].head())

Rows after cleaning: 300000


,combined_text,clean_text
0,Street Condition Pothole The Department of Tra...,street condition pothole the transportation re...
1,Street Condition Pothole The Department of Tra...,street condition pothole the transportation de...
2,Street Condition Pothole The Department of Tra...,street condition pothole the transportation re...
3,Graffiti Graffiti The graffiti on this propert...,graffiti graffiti the graffiti this property h...
4,Street Condition Pothole The Department of Tra...,street condition pothole the transportation de...


In [ ]:
# 7. Create sentence embeddings

from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
model = SentenceTransformer(EMBEDDING_MODEL_NAME)

embeddings = model.encode(
    df["clean_text"].tolist(),
    batch_size=256,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Embedding shape:", embeddings.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1172 [00:00<?, ?it/s]

Embedding shape: (300000, 384)


In [ ]:
# 8. Cluster issue descriptions

from sklearn.cluster import MiniBatchKMeans

kmeans = MiniBatchKMeans(
    n_clusters=NUM_CLUSTERS,
    random_state=RANDOM_STATE,
    batch_size=4096,
    n_init="auto"
)

df["issue_cluster"] = kmeans.fit_predict(embeddings)

print(df["issue_cluster"].value_counts().sort_index())

issue_cluster
0     28885
1     14105
2     38848
3     20244
4     17283
5      9263
6     12772
7      7855
8     20096
9     20300
10    11352
11     8642
12    10615
13     6780
14    22714
15    19599
16     4425
17    10198
18     9790
19     6234
Name: count, dtype: int64


In [ ]:
# 9. Summarize clusters with top terms and example records

from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

vectorizer = TfidfVectorizer(
    max_features=5000,
    min_df=10,
    max_df=0.80,
    ngram_range=(1, 2)
)

X_tfidf = vectorizer.fit_transform(df["clean_text"])
terms = np.array(vectorizer.get_feature_names_out())

summaries = []

for cluster_id in sorted(df["issue_cluster"].unique()):
    idx = df["issue_cluster"].values == cluster_id
    cluster_size = int(idx.sum())

    mean_tfidf = X_tfidf[idx].mean(axis=0).A1
    top_idx = mean_tfidf.argsort()[-12:][::-1]
    top_terms = terms[top_idx].tolist()

    examples = df.loc[idx, "combined_text"].dropna().astype(str).head(5).tolist()

    summaries.append({
        "issue_cluster": cluster_id,
        "record_count": cluster_size,
        "share_of_sample": cluster_size / len(df),
        "top_terms": ", ".join(top_terms),
        "example_1": examples[0] if len(examples) > 0 else "",
        "example_2": examples[1] if len(examples) > 1 else "",
        "suggested_business_label": ""
    })

summary_df = pd.DataFrame(summaries).sort_values("record_count", ascending=False)

display(summary_df)

,issue_cluster,record_count,share_of_sample,top_terms,example_1,example_2,suggested_business_label
2,2,38848,0.129493,"building, heat, hot, hot water, water, heat ho...",HEAT/HOT WATER ENTIRE BUILDING This complaint ...,HEAT/HOT WATER ENTIRE BUILDING The following c...,
0,0,28885,0.096283,"new, police, your, york police, york, new york...",Noise - Street/Sidewalk Loud Music/Party New ...,Noise - Residential Loud Music/Party New York...,
14,14,22714,0.075713,"owner, hpd, attempt, the tenant, tenant, condi...",FLOORING/STAIRS FLOOR The following complaint ...,PAINT/PLASTER CEILING The following complaint ...,
9,9,20300,0.067667,"parking, new, police, york police, new york, y...",Illegal Parking Overnight Commercial Storage ...,Illegal Parking Commercial Overnight Parking ...,
3,3,20244,0.067480,"new, noise, police, york, new york, york polic...",Noise - Residential Banging/Pounding New York...,Noise - Residential Banging/Pounding New York...,
8,8,20096,0.066987,"sanitation, the sanitation, sanitation sidewal...",Graffiti Graffiti The graffiti on this propert...,Food Establishment Rodents/Insects/Garbage De...,
15,15,19599,0.065330,"new, police, york police, york, new york, you,...",Illegal Parking Blocked Sidewalk New York Cit...,Illegal Parking Blocked Sidewalk New York Cit...,
4,4,17283,0.057610,"parks, transportation, light, street light, ho...",Damaged Tree Tree Leaning/Uprooted Department...,Traffic Drag Racing New York City Police Depa...,
1,1,14105,0.047017,"new, police, blocked hydrant, hydrant the, hyd...",Illegal Parking Blocked Hydrant New York City...,Illegal Parking Blocked Hydrant New York City...,
6,6,12772,0.042573,"transportation, street condition, the transpor...",Street Condition Pothole The Department of Tra...,Street Condition Pothole The Department of Tra...,


In [ ]:
# 10. Optional helper labels
# These are starting labels only. Review and edit the CSV manually.

def suggest_label(top_terms):
    text = top_terms.lower()

    rules = [
        ("Heating or hot water issue", ["heat", "hot water", "boiler", "radiator"]),
        ("Noise complaint", ["noise", "loud", "music", "party"]),
        ("Illegal parking or blocked access", ["parking", "blocked", "driveway", "vehicle"]),
        ("Street or sidewalk condition", ["street", "sidewalk", "pothole", "curb"]),
        ("Sanitation or trash issue", ["trash", "garbage", "sanitation", "dirty"]),
        ("Water leak or plumbing issue", ["leak", "water", "plumbing", "sewer"]),
        ("Rodent or pest issue", ["rodent", "rat", "mice", "pest"]),
        ("Building or housing maintenance", ["building", "apartment", "landlord", "maintenance"]),
        ("Tree or park issue", ["tree", "park", "branch"]),
        ("Traffic or signal issue", ["traffic", "signal", "light"])
    ]

    for label, keywords in rules:
        if any(k in text for k in keywords):
            return label

    return "Needs manual review"

summary_df["suggested_business_label"] = summary_df["top_terms"].apply(suggest_label)

display(summary_df)

,issue_cluster,record_count,share_of_sample,top_terms,example_1,example_2,suggested_business_label
2,2,38848,0.129493,"building, heat, hot, hot water, water, heat ho...",HEAT/HOT WATER ENTIRE BUILDING This complaint ...,HEAT/HOT WATER ENTIRE BUILDING The following c...,Heating or hot water issue
0,0,28885,0.096283,"new, police, your, york police, york, new york...",Noise - Street/Sidewalk Loud Music/Party New ...,Noise - Residential Loud Music/Party New York...,Noise complaint
14,14,22714,0.075713,"owner, hpd, attempt, the tenant, tenant, condi...",FLOORING/STAIRS FLOOR The following complaint ...,PAINT/PLASTER CEILING The following complaint ...,Needs manual review
9,9,20300,0.067667,"parking, new, police, york police, new york, y...",Illegal Parking Overnight Commercial Storage ...,Illegal Parking Commercial Overnight Parking ...,Illegal parking or blocked access
3,3,20244,0.067480,"new, noise, police, york, new york, york polic...",Noise - Residential Banging/Pounding New York...,Noise - Residential Banging/Pounding New York...,Noise complaint
8,8,20096,0.066987,"sanitation, the sanitation, sanitation sidewal...",Graffiti Graffiti The graffiti on this propert...,Food Establishment Rodents/Insects/Garbage De...,Street or sidewalk condition
15,15,19599,0.065330,"new, police, york police, york, new york, you,...",Illegal Parking Blocked Sidewalk New York Cit...,Illegal Parking Blocked Sidewalk New York Cit...,Street or sidewalk condition
4,4,17283,0.057610,"parks, transportation, light, street light, ho...",Damaged Tree Tree Leaning/Uprooted Department...,Traffic Drag Racing New York City Police Depa...,Street or sidewalk condition
1,1,14105,0.047017,"new, police, blocked hydrant, hydrant the, hyd...",Illegal Parking Blocked Hydrant New York City...,Illegal Parking Blocked Hydrant New York City...,Illegal parking or blocked access
6,6,12772,0.042573,"transportation, street condition, the transpor...",Street Condition Pothole The Department of Tra...,Street Condition Pothole The Department of Tra...,Street or sidewalk condition


In [ ]:
import os
import joblib
import json

sample_output_path = os.path.join(OUTPUT_DIR, "nlp_issue_category_sample.parquet")
summary_output_path = os.path.join(OUTPUT_DIR, "nlp_issue_cluster_summary.csv")
model_output_path = os.path.join(OUTPUT_DIR, "nlp_issue_kmeans_model.joblib")
config_output_path = os.path.join(OUTPUT_DIR, "nlp_issue_category_config.json")

df.to_parquet(sample_output_path, index=False)
summary_df.to_csv(summary_output_path, index=False)
joblib.dump(kmeans, model_output_path)

config = {
    "available_text_columns": available_text_columns,
    "num_clusters": NUM_CLUSTERS,
    "embedding_model_name": EMBEDDING_MODEL_NAME
}

with open(config_output_path, "w") as f:
    json.dump(config, f, indent=2)

print("Saved to:", OUTPUT_DIR)

Saved to: /content/nlp_issue_outputs


In [ ]:
# ZIP ALL OUTPUTS FOR DOWNLOAD


import shutil

ZIP_PATH = "/content/nlp_issue_outputs.zip"

shutil.make_archive(
    base_name=ZIP_PATH.replace(".zip", ""),
    format="zip",
    root_dir=OUTPUT_DIR
)

print("ZIP created at:", ZIP_PATH)